# Adaptive uncertainty-aware planning для Cosmos Policy

**Frozen confirmatory snapshot · 13 August 2026**

Отдельный notebook для новой линии экспериментов. Screening, независимый
confirmatory split и качественные video replays здесь явно разделены. Старый
`ysda_world_models.ipynb` не используется как источник итоговых чисел.

## 1. Постановка

На каждом query из одного реального наблюдения генерируются $N=4$
stochastic candidates. Для кандидата $i$ доступны predicted value $V_i$,
action chunk $A_i\in\mathbb{R}^{16\times7}$, future image/proprio и latent
copies action/value.

Каждый candidate является одной согласованной авторегрессионной веткой
$A_i\rightarrow \widehat s_i\rightarrow V_i$: value относится к action и
future-state того же sample. Четыре ветки можно считать батчем на GPU, но при
planning они остаются четырьмя отдельными кандидатами, а не одной усреднённой
value-матрицей.

Baseline выбирает

$$
i_{V}=\arg\max_i V_i.
$$

Для первого действия uncertainty кандидата агрегируется по повторным latent
copies $k$ и всем семи координатам action $d$:

$$
U_i=\frac{1}{B}\sum_{b=1}^{B}
\sqrt{\sum_{d=1}^{7}
\operatorname{Std}_{k}\!\left[\widetilde A_{i,b,k,0,d}\right]^2},
\qquad
S_i=z(V_i)-\lambda z(U_i),
\qquad i_R=\arg\max_i S_i.
$$

Здесь учитываются все 7 координат первого действия, включая gripper; $B=1$ в
обычном online rollout. Повторные latent copies находятся внутри одного
candidate и отличаются от четырёх stochastic candidates $i$.
Стандартизация выполняется отдельно внутри текущего query:

$$
z(x_i)=\frac{x_i-\frac1N\sum_jx_j}
{\sqrt{\frac1N\sum_j(x_j-\bar x)^2}+10^{-6}}.
$$

`phase_l1_r0.3` использует $i_R$ только при $t/T_{max}\le0.3$, затем
возвращается к $i_V$. `requery_l1_h8` всегда выбирает $i_R$, но исполняет
8 вместо 16 действий, когда $i_R\ne i_V$, и раньше получает новое реальное
наблюдение.

## 2. Протокол и гипотезы

1. Screening: 3 известных LIBERO-PRO boundary case, 12 одинаковых seed.
2. Выбор одного метода без extra inference и одного adaptive-horizon метода по
   заранее заданному критерию

$$
J=\Delta_{pool}+0.5\min_c\Delta_c
-0.02\max(0,Q_{ratio}-1).
$$

3. Frozen confirmatory: 30 новых seed на каждом из 6 case, включая 3 новых OOD
   holdout; всего 180 paired seed на стратегию.
4. Primary endpoint: paired success delta против `max(value)`, stratified
   bootstrap CI, exact McNemar и Holm correction для двух frozen гипотез.
5. Exact replay измеряет noise floor. Видео выбираются после статистики только
   по discordant outcome и не используются для success-rate оценки.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import HTML, Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'experiments':
    PROJECT_ROOT = PROJECT_ROOT.parent

CALIBRATION = PROJECT_ROOT / 'experiments/campaigns/adaptive_screening_20260813/analysis/adaptive_summary'
CONFIRMATORY = PROJECT_ROOT / 'experiments/campaigns/adaptive_confirmatory_20260813/analysis/adaptive_summary'
MEDIA = PROJECT_ROOT / 'experiments/final_results_media/adaptive_confirmatory_20260813'

frozen = pd.read_csv(CONFIRMATORY / 'frozen_confirmatory_results.csv')
per_case = pd.read_csv(CONFIRMATORY / 'paired_by_case.csv')
replay = pd.read_csv(CONFIRMATORY / 'replay_control_summary.csv')
print(f'Loaded {int(frozen.paired_rollouts.max())} paired seeds per frozen strategy.')

## 3. Calibration selection

| Category           | Selected strategy   | max(value)   | Result   | Delta    | Worst case   | Query cost   |   Selection utility J |
|:-------------------|:--------------------|:-------------|:---------|:---------|:-------------|:-------------|----------------------:|
| adaptive_horizon   | requery_l1_h8       | 18/36        | 29/36    | +30.6 pp | +16.7 pp     | 1.26x        |              0.383616 |
| no_extra_inference | phase_l1_r0.3       | 18/36        | 25/36    | +19.4 pp | +8.3 pp      | 1.00x        |              0.236111 |

![Calibration success/compute trade-off](campaigns/adaptive_screening_20260813/analysis/adaptive_summary/plots/success_compute_tradeoff.png)

Это screening-числа, а не финальная оценка эффекта.

## 4. Frozen confirmatory result

| Category           | Strategy      | max(value)   | Result   | delta    | 95% CI              | W/L/T     |   Holm p | query cost   | actual calls   |
|:-------------------|:--------------|:-------------|:---------|:---------|:--------------------|:----------|---------:|:-------------|:---------------|
| adaptive_horizon   | requery_l1_h8 | 115/180      | 143/180  | +15.6 pp | [+7.2 pp; +23.3 pp] | 46/18/116 |   0.0012 | 1.27x        | 1.07x          |
| no_extra_inference | phase_l1_r0.3 | 115/180      | 130/180  | +8.3 pp  | [+0.6 pp; +16.1 pp] | 33/18/129 |   0.0489 | 1.00x        | 0.95x          |

![Confirmatory pooled delta](campaigns/adaptive_confirmatory_20260813/analysis/adaptive_summary/plots/pooled_strategy_delta.png)

`query cost` нормирован на число query, необходимое для фактически исполненного
числа env steps при стандартном chunk=16. `actual calls` может быть меньше этой
оценки, если улучшенная стратегия раньше завершает эпизод.

## 5. Перенос и неоднородность

### По каждому case

| Case                                     | Method        | Baseline   | Strategy   | Delta    |   Wins |   Losses |
|:-----------------------------------------|:--------------|:-----------|:-----------|:---------|-------:|---------:|
| goal_mug_task9_init0_adaptive_confirm    | phase_l1_r0.3 | 28/30      | 29/30      | +3.3 pp  |      1 |        0 |
| goal_mug_task9_init0_adaptive_confirm    | requery_l1_h8 | 28/30      | 27/30      | -3.3 pp  |      2 |        3 |
| long_milk_task9_init0_adaptive_confirm   | phase_l1_r0.3 | 21/30      | 23/30      | +6.7 pp  |      8 |        6 |
| long_milk_task9_init0_adaptive_confirm   | requery_l1_h8 | 21/30      | 27/30      | +20.0 pp |      8 |        2 |
| long_mug_task4_init0_adaptive_confirm    | phase_l1_r0.3 | 23/30      | 24/30      | +3.3 pp  |      4 |        3 |
| long_mug_task4_init0_adaptive_confirm    | requery_l1_h8 | 23/30      | 29/30      | +20.0 pp |      6 |        0 |
| milk_task5_init0_adaptive_confirm        | phase_l1_r0.3 | 14/30      | 18/30      | +13.3 pp |     10 |        6 |
| milk_task5_init0_adaptive_confirm        | requery_l1_h8 | 14/30      | 11/30      | -10.0 pp |      7 |       10 |
| spatial_mug_task0_init0_adaptive_confirm | phase_l1_r0.3 | 22/30      | 24/30      | +6.7 pp  |      5 |        3 |
| spatial_mug_task0_init0_adaptive_confirm | requery_l1_h8 | 22/30      | 28/30      | +20.0 pp |      7 |        1 |
| yellow_task8_init0_adaptive_confirm      | phase_l1_r0.3 | 7/30       | 12/30      | +16.7 pp |      5 |        0 |
| yellow_task8_init0_adaptive_confirm      | requery_l1_h8 | 7/30       | 21/30      | +46.7 pp |     16 |        2 |

![Per-case paired delta](campaigns/adaptive_confirmatory_20260813/analysis/adaptive_summary/plots/paired_delta_heatmap.png)

### Известные boundary против новых OOD holdout

| Stratum         | Strategy      |   Paired seeds | Delta    | 95% CI              |
|:----------------|:--------------|---------------:|:---------|:--------------------|
| known_boundary  | requery_l1_h8 |             90 | +18.9 pp | [+6.7 pp; +31.1 pp] |
| known_boundary  | phase_l1_r0.3 |             90 | +11.1 pp | [+0.0 pp; +22.2 pp] |
| new_ood_holdout | requery_l1_h8 |             90 | +12.2 pp | [+2.2 pp; +22.2 pp] |
| new_ood_holdout | phase_l1_r0.3 |             90 | +5.6 pp  | [-4.4 pp; +15.6 pp] |

## 6. Контроли и noise floor

| Reference method   | Exact replay     | Reference   | Replay   | Disagreement   |
|:-------------------|:-----------------|:------------|:---------|:---------------|
| max_value          | max_value_replay | 63.9%       | 65.0%    | 8/180 (4.4%)   |
| action_l1          | action_l1_replay | 72.8%       | 71.1%    | 17/180 (9.4%)  |

Fixed `action_l1` сравнивается с теми же 180 baseline seed. Его результат:
**+8.9 pp**, CI
[+0.6 pp; +17.2 pp],
exact McNemar p=0.0559.

Outcome noise floor равен 4.4% для exact `max(value)` replay и 9.4% для
`action_l1`. Поэтому качественное расхождение одного видео само по себе не
доказывает эффект; основной вывод опирается на все paired seeds.

## 7. Mechanism diagnostics

Эти таблицы exploratory и не меняют frozen planner.

![Early failure AUROC](campaigns/adaptive_confirmatory_20260813/analysis/adaptive_summary/plots/early_failure_predictor_auc.png)

| feature                                                      |   episodes |   failures |   cases |   raw_auc_high_predicts_fail |   case_controlled_auc_high_predicts_fail |   case_controlled_oriented_auc | risk_direction   |
|:-------------------------------------------------------------|-----------:|-----------:|--------:|-----------------------------:|-----------------------------------------:|-------------------------------:|:-----------------|
| value_std__max                                               |        180 |         65 |       6 |                     0.622475 |                                 0.588227 |                       0.588227 | high             |
| candidate_value_mean__mean                                   |        180 |         65 |       6 |                     0.62796  |                                 0.58796  |                       0.58796  | high             |
| candidate_value_mean__max                                    |        180 |         65 |       6 |                     0.628094 |                                 0.586756 |                       0.586756 | high             |
| candidate_value_mean__delta                                  |        180 |         65 |       6 |                     0.637324 |                                 0.579532 |                       0.579532 | high             |
| value_range__max                                             |        180 |         65 |       6 |                     0.618462 |                                 0.572977 |                       0.572977 | high             |
| value_std__mean                                              |        180 |         65 |       6 |                     0.615652 |                                 0.552776 |                       0.552776 | high             |
| value_range__mean                                            |        180 |         65 |       6 |                     0.614783 |                                 0.546221 |                       0.546221 | high             |
| latent_action_copy_std_mean_mean_over_samples__mean          |        180 |         65 |       6 |                     0.47893  |                                 0.546087 |                       0.546087 | high             |
| latent_action_first_step_copy_l2_std_mean_over_samples__mean |        180 |         65 |       6 |                     0.50301  |                                 0.456455 |                       0.543545 | low              |
| candidate_action_internal_consistency_mean__mean             |        180 |         65 |       6 |                     0.50301  |                                 0.456455 |                       0.543545 | low              |

![Uncertainty vs prediction error](campaigns/adaptive_confirmatory_20260813/analysis/adaptive_summary/plots/uncertainty_prediction_error_correlations.png)

| online_metric                                          | prediction_error                   |   queries |   cases |   raw_spearman |   case_controlled_rank_correlation |
|:-------------------------------------------------------|:-----------------------------------|----------:|--------:|---------------:|-----------------------------------:|
| candidate_value_mean                                   | prediction_error_future_proprio_l2 |      2570 |       6 |       0.514164 |                           0.552104 |
| latent_action_copy_std_mean_mean_over_samples          | prediction_error_future_proprio_l2 |      2570 |       6 |       0.629131 |                           0.548414 |
| candidate_action_internal_consistency_mean             | prediction_error_future_proprio_l2 |      2570 |       6 |       0.566997 |                           0.461611 |
| latent_action_first_step_copy_l2_std_mean_over_samples | prediction_error_future_proprio_l2 |      2570 |       6 |       0.566997 |                           0.461611 |
| candidate_value_mean                                   | prediction_error_future_wrist_mse  |      2570 |       6 |       0.41819  |                           0.424381 |
| latent_value_element_std_mean_mean_over_samples        | prediction_error_future_proprio_l2 |      2570 |       6 |       0.433901 |                           0.385455 |
| candidate_value_mean                                   | prediction_error_future_image_mse  |      2570 |       6 |       0.133878 |                           0.36718  |
| candidate_action_consensus_first_mean                  | prediction_error_future_proprio_l2 |      2570 |       6 |       0.426728 |                           0.347136 |
| action_first_step_l2_std                               | prediction_error_future_proprio_l2 |      2570 |       6 |       0.414521 |                           0.331153 |
| latent_action_copy_std_mean_mean_over_samples          | prediction_error_future_image_mse  |      2570 |       6 |       0.320195 |                           0.254698 |

## 8. Matched-seed video replays

Видео выбраны механически по discordant confirmatory outcomes. В каждой группе
совпадают suite/task/init/rollout seed; меняется только planning strategy.
Исходная selected-strategy пара полностью воспроизвела оба бинарных outcome в
**5/8** exact replays. Поэтому ролики
показывают механизм расхождения траекторий, а causal success-rate вывод берётся
из всех 180 paired seeds выше.

<h4>goal_mug_task9_init0_adaptive_confirm, rollout_seed=760388</h4><p>selected pair changed under exact replay</p><table style="width:100%"><tr><td style="vertical-align:top;padding:8px"><b>action_l1</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=320<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed760388__action_l1__fail.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed760388__action_l1__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>max_value</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=137<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed760388__max_value__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed760388__max_value__success.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>phase_l1_r0.3</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=320<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed760388__phase_l1_r0.3__fail.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed760388__phase_l1_r0.3__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>requery_l1_h8</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=162<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed760388__requery_l1_h8__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed760388__requery_l1_h8__success.mp4">Open MP4</a></td></tr></table>
<h4>goal_mug_task9_init0_adaptive_confirm, rollout_seed=760485</h4><p>selected pair changed under exact replay</p><table style="width:100%"><tr><td style="vertical-align:top;padding:8px"><b>action_l1</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=160<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed760485__action_l1__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed760485__action_l1__success.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>max_value</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=149<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed760485__max_value__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed760485__max_value__success.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>phase_l1_r0.3</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=160<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed760485__phase_l1_r0.3__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed760485__phase_l1_r0.3__success.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>requery_l1_h8</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=134<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed760485__requery_l1_h8__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed760485__requery_l1_h8__success.mp4">Open MP4</a></td></tr></table>
<h4>goal_mug_task9_init0_adaptive_confirm, rollout_seed=761552</h4><p>selected pair reproduced</p><table style="width:100%"><tr><td style="vertical-align:top;padding:8px"><b>action_l1</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=168<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed761552__action_l1__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed761552__action_l1__success.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>max_value</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=320<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed761552__max_value__fail.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed761552__max_value__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>phase_l1_r0.3</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=168<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed761552__phase_l1_r0.3__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed761552__phase_l1_r0.3__success.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>requery_l1_h8</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=320<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed761552__requery_l1_h8__fail.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/goal_mug_task9_init0_adaptive_confirm__seed761552__requery_l1_h8__fail.mp4">Open MP4</a></td></tr></table>
<h4>long_milk_task9_init0_adaptive_confirm, rollout_seed=750582</h4><p>selected pair reproduced</p><table style="width:100%"><tr><td style="vertical-align:top;padding:8px"><b>action_l1</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=520<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/long_milk_task9_init0_adaptive_confirm__seed750582__action_l1__fail.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/long_milk_task9_init0_adaptive_confirm__seed750582__action_l1__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>max_value</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=260<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/long_milk_task9_init0_adaptive_confirm__seed750582__max_value__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/long_milk_task9_init0_adaptive_confirm__seed750582__max_value__success.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>phase_l1_r0.3</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=520<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/long_milk_task9_init0_adaptive_confirm__seed750582__phase_l1_r0.3__fail.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/long_milk_task9_init0_adaptive_confirm__seed750582__phase_l1_r0.3__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>requery_l1_h8</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=244<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/long_milk_task9_init0_adaptive_confirm__seed750582__requery_l1_h8__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/long_milk_task9_init0_adaptive_confirm__seed750582__requery_l1_h8__success.mp4">Open MP4</a></td></tr></table>
<h4>long_mug_task4_init0_adaptive_confirm, rollout_seed=730000</h4><p>selected pair reproduced</p><table style="width:100%"><tr><td style="vertical-align:top;padding:8px"><b>action_l1</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=225<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed730000__action_l1__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed730000__action_l1__success.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>max_value</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=520<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed730000__max_value__fail.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed730000__max_value__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>phase_l1_r0.3</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=225<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed730000__phase_l1_r0.3__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed730000__phase_l1_r0.3__success.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>requery_l1_h8</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=216<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed730000__requery_l1_h8__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed730000__requery_l1_h8__success.mp4">Open MP4</a></td></tr></table>
<h4>long_mug_task4_init0_adaptive_confirm, rollout_seed=730776</h4><p>selected pair reproduced</p><table style="width:100%"><tr><td style="vertical-align:top;padding:8px"><b>action_l1</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=227<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed730776__action_l1__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed730776__action_l1__success.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>max_value</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=520<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed730776__max_value__fail.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed730776__max_value__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>phase_l1_r0.3</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=227<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed730776__phase_l1_r0.3__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed730776__phase_l1_r0.3__success.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>requery_l1_h8</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=216<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed730776__requery_l1_h8__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed730776__requery_l1_h8__success.mp4">Open MP4</a></td></tr></table>
<h4>long_mug_task4_init0_adaptive_confirm, rollout_seed=731067</h4><p>selected pair changed under exact replay</p><table style="width:100%"><tr><td style="vertical-align:top;padding:8px"><b>action_l1</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=520<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed731067__action_l1__fail.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed731067__action_l1__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>max_value</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=520<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed731067__max_value__fail.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed731067__max_value__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>phase_l1_r0.3</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=520<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed731067__phase_l1_r0.3__fail.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed731067__phase_l1_r0.3__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>requery_l1_h8</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=219<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed731067__requery_l1_h8__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/long_mug_task4_init0_adaptive_confirm__seed731067__requery_l1_h8__success.mp4">Open MP4</a></td></tr></table>
<h4>milk_task5_init0_adaptive_confirm, rollout_seed=710194</h4><p>selected pair reproduced</p><table style="width:100%"><tr><td style="vertical-align:top;padding:8px"><b>action_l1</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=220<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/milk_task5_init0_adaptive_confirm__seed710194__action_l1__fail.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/milk_task5_init0_adaptive_confirm__seed710194__action_l1__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>max_value</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=145<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/milk_task5_init0_adaptive_confirm__seed710194__max_value__success.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/milk_task5_init0_adaptive_confirm__seed710194__max_value__success.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>phase_l1_r0.3</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=220<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/milk_task5_init0_adaptive_confirm__seed710194__phase_l1_r0.3__fail.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/milk_task5_init0_adaptive_confirm__seed710194__phase_l1_r0.3__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px"><b>requery_l1_h8</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=220<br><video controls preload="metadata" width="270" src="final_results_media/adaptive_confirmatory_20260813/milk_task5_init0_adaptive_confirm__seed710194__requery_l1_h8__fail.mp4"></video><br><a href="final_results_media/adaptive_confirmatory_20260813/milk_task5_init0_adaptive_confirm__seed710194__requery_l1_h8__fail.mp4">Open MP4</a></td></tr></table>

## 9. Выводы

- **`requery_l1_h8`:** +15.6 pp против `max(value)`, CI [+7.2 pp; +23.3 pp], Holm p=0.0012; гипотеза подтвердилась после Holm-коррекции.
- **`phase_l1_r0.3`:** +8.3 pp против `max(value)`, CI [+0.6 pp; +16.1 pp], Holm p=0.0489; гипотеза подтвердилась после Holm-коррекции.
- **Fixed `action_l1` control:** +8.9 pp, CI [+0.6 pp; +17.2 pp], McNemar p=0.0559.
- **Requery против fixed penalty:** +6.7 pp, CI [-1.1 pp; +13.9 pp], McNemar p=0.1263; преимущество над `max(value)` подтверждено, но над сильным fixed control пока нет.

Дополнительно:

- эффекты ожидаемо уменьшились относительно screening (`requery`: +30.6 до
  +15.6 п.п.; `phase`: +19.4 до +8.3 п.п.), но сохранили направление на новых
  seed; это показывает, зачем calibration и confirmatory split были разделены;
- `phase_l1_r0.3` дал положительный delta во всех шести cases и не требует
  дополнительных calls, но практически не отличается от fixed `action_l1`;
- `requery_l1_h8` улучшил четыре cases, однако ухудшил `milk_task5` на 10 п.п.
  и `goal_mug_task9` на 3.3 п.п.; нужен context gate, а не безусловный requery;
- pooled requery effect сохранился отдельно на известных boundary cases
  (+18.9 п.п.) и новых OOD holdout (+12.2 п.п.);
- ранние uncertainty/value признаки дают лишь умеренный fail AUROC (лучший
  case-controlled результат 0.588), поэтому они лучше подходят для
  относительного candidate ranking, чем для общего порога fail;
- latent-action uncertainty коррелирует с next-chunk proprio error
  (case-controlled Spearman 0.548), что поддерживает предполагаемый механизм;
- adaptive requery оценивается вместе с query-cost, а не как бесплатное улучшение;
- prediction error после chunk является диагностикой, но недоступен для выбора
  того же chunk без отдельного learned surrogate;
- следующий шаг -- learned task/phase gate с leave-one-suite-out проверкой,
  который сохраняет requery там, где он помогает, и отключает на вредных cases.

## 10. Воспроизводимость

- [Гипотезы и frozen protocol](ADAPTIVE_PLANNING_HYPOTHESES_20260813.md)
- [Calibration analysis](campaigns/adaptive_screening_20260813/analysis/adaptive_summary/README.md)
- [Confirmatory analysis](campaigns/adaptive_confirmatory_20260813/analysis/adaptive_summary/README.md)
- [Frozen campaign config](configs/libero_campaign_adaptive_confirmatory_frozen.json)
- [Видео и manifest](final_results_media/adaptive_confirmatory_20260813/README.md)